In [ ]:
# @title 1. Import Required Libraries

import random
import csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import cv2
from ultralytics import YOLO, solutions

In [ ]:
cap = cv2.VideoCapture("videos/d14_right.mp4")

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("speed_management.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

CAMERA_ANCHOR_POINT = [w/2, h, w/2, h]

model = YOLO("yolo26n.pt") # Replace with your custom weights if needed

distance_calculator = solutions.DistanceCalculation(
    show=True,  # whether to display the distance calculation results on the frame.
    model="yolo26n.pt",  # path to the YOLOv8 model file.
    fps=fps,  # adjust speed based on frame per second
    # max_speed=120,  # cap speed to a max value (km/h) to avoid outliers
    # max_hist=5,  # minimum frames object tracked before computing speed
    # meter_per_pixel=0.05,  # highly depends on the camera configuration
    # classes=[0, 2],  # estimate speed of specific classes.
    # line_width=2,  # adjust the line width for bounding boxes
)


In [ ]:

i = 0

while cap.isOpened():
    success, im0 = cap.read()
    i+=1
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    tracks = model.track(im0, persist=True, verbose=False)
    for r in tracks:
        for box, track_id in zip(r.boxes, r.boxes.id):
            if len(distance_calculator.selected_boxes)>1:
                distance_calculator.selected_boxes = {-1: CAMERA_ANCHOR_POINT,
                                                        int(track_id): box.xyxy[0].tolist()}
            else:
                distance_calculator.selected_boxes[int(track_id)] = box.xyxy[0].tolist()
            results = distance_calculator.process(im0)
            cv2.waitKey(0)

    # video_writer.write(annotated_frame)  # write the processed frame.
    # print(f"\rFrame {i} processed.", end='')



cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows